# PART 04 · 2장 가설검정

교재 316~335쪽 본문 소스 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

#### 1. 검정 유형을 고르는 첫 기준

#### 2. 공통 라이브러리와 데이터 불러오기

다음 코드는 이 장에서 사용할 라이브러리와 네 개의 데이터를 한 번만 불러온다.

In [1]:
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
study_path = (
    'data/part04_type3/one_sample_ttest_study_time/'
    'study_time.csv'
)
app_path = (
    'data/part04_type3/independent_ttest_app_time/'
    'app_task_time.csv'
)
paired_path = (
    'data/part04_type3/paired_ttest_training_score/'
    'training_score.csv'
)
trial_path = (
    'data/part04_type3/logistic_regression_trial_signup/'
    'trial_signup.csv'
)
study_df = pd.read_csv(study_path)
app_df = pd.read_csv(app_path)
paired_df = pd.read_csv(paired_path)
trial_df = pd.read_csv(trial_path)
print(study_df.shape, app_df.shape)
print(paired_df.shape, trial_df.shape)

(36, 2) (78, 3)
(34, 3) (520, 7)


## 01 가설검정

## 02 단일표본 t-검정

### 1 사용 상황과 가설

#### 1. 사용하는 상황과 가설 방향

### 2 검정 실행과 결과 확인

#### 1. 데이터와 기본 코드

study_df의 student_id는 수험생을 구분하는 식별자이고, study_hours는 주간 학습 시간이다. 평균 검정에는 study_hours만 사용한다.

In [2]:
x = study_df['study_hours']
popmean = 40
alpha = 0.05
result = stats.ttest_1samp(
    x,
    popmean=popmean,
    alternative='two-sided'
)
print('sample_size: ', len(x))
print('sample_mean: ', round(x.mean(), 4))
print('sample_std: ', round(x.std(), 4))
print('t_statistic: ', round(result.statistic, 4))
print('p_value: ', format(result.pvalue, '.6f'))
if result.pvalue < alpha:
    print('decision: 귀무가설 기각')
else:
    print('decision: 귀무가설 기각하지 못함')

sample_size:  36
sample_mean:  43.4028
sample_std:  4.5477
t_statistic:  4.4895
p_value:  0.000074
decision: 귀무가설 기각


#### 2. 결과 객체에서 필요한 값 꺼내기

## 03 독립표본 t-검정

### 1 데이터 준비

#### 1. 집단과 분석 컬럼 분리

두 집단의 데이터를 각각 분리하고 요약값을 확인해본다.

In [3]:
a = app_df.loc[
    app_df['app_version'] == 'A',
    'completion_minutes'
]
b = app_df.loc[
    app_df['app_version'] == 'B',
    'completion_minutes'
]
print('A_count: ', len(a))
print('B_count: ', len(b))
print('A_mean: ', round(a.mean(), 4))
print('B_mean: ', round(b.mean(), 4))
print('A_std: ', round(a.std(), 4))
print('B_std: ', round(b.std(), 4))

A_count:  38
B_count:  40
A_mean:  30.1368
B_mean:  26.1225
A_std:  5.3754
B_std:  5.553


### 2 검정 실행과 옵션

#### 1. Welch 독립표본 t-검정

이번 항목에서는 두 집단의 분산이 같다고 단정하지 않는 Welch 방식으로 먼저 검정한다. equal_var=False를 지정하면 Welch 독립표본 t-검정이 실행된다.

In [4]:
result = stats.ttest_ind(
    a,
    b,
    equal_var=False,
    alternative='two-sided'
)
print('t_statistic: ', format(result.statistic, '.4f'))
print('p_value: ', format(result.pvalue, '.6f'))
print('df: ', round(result.df, 4))

t_statistic:  3.2440
p_value:  0.001753
df:  75.9712


#### 2. equal_var 옵션

## 04 대응표본 t-검정

### 1 차이값 만들기

#### 1. 차이값 만들기

대응표본 t-검정은 같은 대상의 전후 차이를 본다. 차이값의 요약부터 확인해본다.

In [5]:
before = paired_df['before_score']
after = paired_df['after_score']
diff = after-before
print('sample_size: ', len(diff))
print('before_mean: ', round(before.mean(), 4))
print('after_mean: ', round(after.mean(), 4))
print('diff_mean: ', round(diff.mean(), 4))
print('diff_std: ', round(diff.std(), 4))

sample_size:  34
before_mean:  69.9941
after_mean:  75.4971
diff_mean:  5.5029
diff_std:  2.1187


### 2 검정 실행과 방향 확인

#### 1. 양측 대응표본 t-검정

사후 점수와 사전 점수로 양측 대응표본 t-검정을 실행해본다.

In [6]:
result = stats.ttest_rel(
    after,
    before,
    alternative='two-sided'
)
print('t_statistic: ', format(result.statistic, '.4f'))
print('p_value: ', format(result.pvalue, '.6e'))

t_statistic:  15.1450
p_value:  2.042465e-16


#### 2. 인수 순서와 단측검정 방향

‘교육 후 점수가 교육 전보다 높아졌는가?’를 검정하려면 after - before가 0보다 큰지를 확인해야 한다. 따라서 after를 첫 번째 인수로 두고 alternative=‘greater’를 사용한다.

In [7]:
right_result = stats.ttest_rel(
    after,
    before,
    alternative='greater'
)
print('t_statistic: ', format(right_result.statistic, '.4f'))
print('p_value: ', format(right_result.pvalue, '.6e'))

t_statistic:  15.1450
p_value:  1.021233e-16


## 05 정규성 검정

### 1 Shapiro-Wilk 검정

두 집단 각각에 Shapiro-Wilk 정규성 검정을 적용해본다.

In [8]:
a_normal = stats.shapiro(a)
b_normal = stats.shapiro(b)
print('A_statistic: ', round(a_normal.statistic, 4))
print('A_p_value: ', format(a_normal.pvalue, '.6f'))
print('B_statistic: ', round(b_normal.statistic, 4))
print('B_p_value: ', format(b_normal.pvalue, '.6f'))

A_statistic:  0.9552
A_p_value:  0.131846
B_statistic:  0.9529
B_p_value:  0.095337


### 2 대응표본에서는 차이값을 확인

대응표본 t-검정의 정규성 가정은 before와 after를 따로 보는 것이 아니라 두 측정값의 차이 diff가 정규분포를 따르는지 확인하는 것이 핵심이다.

In [9]:
diff_normal = stats.shapiro(diff)
print('diff_statistic: ', round(diff_normal.statistic, 4))
print('diff_p_value: ', format(diff_normal.pvalue, '.6f'))

diff_statistic:  0.9423
diff_p_value:  0.071966


## 06 등분산성 검정

### 1 Levene 검정과 활용

#### 1. 표본분산과 Levene 검정

두 집단의 표본분산을 확인하고 Levene 검정으로 등분산성을 검정해본다.

In [10]:
levene_result = stats.levene(a, b)
print('A_variance: ', round(a.var(), 4))
print('B_variance: ', round(b.var(), 4))
print('levene_statistic: ', round(levene_result.statistic, 4))
print('levene_p_value: ', format(levene_result.pvalue, '.6f'))

A_variance:  28.8948
B_variance:  30.8361
levene_statistic:  0.2654
levene_p_value:  0.607938


#### 2. Levene 결과에 따라 equal_var 선택

Levene 검정 결과에 따라 equal_var 값을 정해 t-검정을 이어서 실행해본다.

In [11]:
alpha = 0.05
equal_var_option = (levene_result.pvalue >= alpha)
t_result = stats.ttest_ind(a, b, equal_var=equal_var_option)
print('equal_var: ', equal_var_option)
print('t_statistic: ', round(t_result.statistic, 4))
print('p_value: ', format(t_result.pvalue, '.6f'))

equal_var:  True
t_statistic:  3.2413
p_value:  0.001767


### 2 Bartlett 검정과 비교

#### 1. Levene 검정과 Bartlett 검정

## 07 비율 검정

### 1 표본 비율 확인

#### 1. 성공 개수와 표본 비율

성공 개수와 표본 수, 표본 비율부터 확인해본다.

In [12]:
success_count = int(
    trial_df['signed_up'].sum()
)
nobs = len(trial_df)
sample_prop = success_count/nobs
print('success_count: ', success_count)
print('nobs: ', nobs)
print('sample_prop: ', round(sample_prop, 4))

success_count:  71
nobs:  520
sample_prop:  0.1365


### 2 비율 z 검정 실행

#### 1. 한 집단 비율 z 검정

statsmodels의 proportions_ztest( )에서 count는 성공 개수, nobs는 전체 표본 수, value는 귀무가설의 기준 비율이다. 다음 결과는 statsmodels의 기본 설정으로 계산한 값이며, 문제에서 별도의 공식이나 분산 기준을 지정하면 그 조건을 따른다.

In [13]:
z_stat, p_value = proportions_ztest(
    count=success_count,
    nobs=nobs,
    value=0.10,
    alternative='two-sided'
)
print('z_statistic: ', round(z_stat, 4))
print('p_value: ', format(p_value, '.6f'))

z_statistic:  2.4266
p_value:  0.015240


#### 2. alternative 값에 주의

## 08 두 집단 비율 차이 검정

### 1 집단별 데이터 확인

#### 1. 집단별 성공 개수와 표본 수

집단별 성공 개수와 표본 수를 요약해 본다.

In [14]:
summary = trial_df.groupby(
    'email_opened'
)['signed_up'].agg(
    ['sum', 'count', 'mean']
)
print(summary)

              sum  count      mean
email_opened                      
0              19    226  0.084071
1              52    294  0.176871


### 2 두 집단 z 검정 실행

#### 1. 두 집단 비율 z 검정

요약값을 꺼내 두 집단 비율 z 검정을 실행해본다.

In [15]:
opened_success = int(summary.loc[1, 'sum'])
closed_success = int(summary.loc[0, 'sum'])
opened_n = int(summary.loc[1, 'count'])
closed_n = int(summary.loc[0, 'count'])
count = [opened_success, closed_success]
nobs = [opened_n, closed_n]
z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    value=0,
    alternative='two-sided'
)
opened_rate = opened_success/opened_n
closed_rate = closed_success/closed_n
print('opened_rate: ', round(opened_rate, 4))
print('closed_rate: ', round(closed_rate, 4))
print('rate_diff: ', round(opened_rate-closed_rate, 4))
print('z_statistic: ', round(z_stat, 4))
print('p_value: ', format(p_value, '.6f'))

opened_rate:  0.1769
closed_rate:  0.0841
rate_diff:  0.0928
z_statistic:  3.0551
p_value:  0.002250


#### 2. count와 nobs의 순서

## 09 검정 결과 해석

### 1 기준값 비교

#### 1. 기준값 40과 42 비교

기준값 40과 42에 대한 검정을 나란히 실행해본다.

In [16]:
x = study_df['study_hours']
result_40 = stats.ttest_1samp(
    x,
    popmean=40,
    alternative='two-sided'
)
result_42 = stats.ttest_1samp(
    x,
    popmean=42,
    alternative='two-sided'
)
print('mean: ', round(x.mean(), 4))
print('t_40: ', round(result_40.statistic, 4))
print('p_40: ', format(result_40.pvalue, '.6f'))
print('t_42: ', round(result_42.statistic, 4))
print('p_42: ', format(result_42.pvalue, '.6f'))

mean:  43.4028
t_40:  4.4895
p_40:  0.000074
t_42:  1.8508
p_42:  0.072659


### 2 해석 단계와 표현

#### 1. 결과 해석의 6단계

#### 2. 피해야 할 표현

## 10 SciPy를 활용한 가설검정

### 1 주요 함수와 결과 객체

#### 1. 주요 SciPy 함수

#### 2. 결과 객체 확인

결과 객체의 값을 가공 없이 그대로 출력하면 다음과 같은 형태다.

In [17]:
x = study_df['study_hours']
result = stats.ttest_1samp(
    x,
    popmean=40
)
normality = stats.shapiro(x)
print('t_statistic: ', format(float(result.statistic), '.15f'))
print('t_p_value: ', format(float(result.pvalue), '.15e'))
print('t_df: ', int(len(x)-1))
print('shapiro_statistic: ', format(float(normality.statistic), '.16f'))
print('shapiro_p_value: ', format(float(normality.pvalue), '.17f'))

t_statistic:  4.489465664875056
t_p_value:  7.418465018622554e-05
t_df:  35
shapiro_statistic:  0.9577222540283368
shapiro_p_value:  0.18270363988573918


### 2 공통 사용형

#### 1. 함수 선택과 결과 처리의 공통형

함수 선택과 결과 처리의 공통 형태를 정리하면 다음과 같다.

> 설명용 코드입니다. 그대로 실행되지 않습니다.

```python
result = stats.검정함수(데이터, 옵션)
print(result.statistic)
print(result.pvalue)
if result.pvalue < alpha:
    print('귀무가설 기각')
else:
    print('귀무가설을 기각하지 못함')
```

## 11 결괏값 반올림 처리

### 1 반올림 함수 구분

#### 1. round( ), format( ), f-string 구분

### 2 결괏값 정리와 표기

#### 1. 실제 결괏값 정리

실제 결괏값을 방법별로 정리해 출력해본다.

In [18]:
x = study_df['study_hours']
result = stats.ttest_1samp(
    x,
    popmean=40
)
ci = result.confidence_interval(
    confidence_level=0.95
)
print('mean_raw: ', x.mean())
print('mean_round2: ', round(x.mean(), 2))
print('mean_format2: ', format(x.mean(), '.2f'))
print('t_raw: ', result.statistic)
print('t_round4: ', round(result.statistic, 4))
print('p_raw: ', result.pvalue)
print('p_format6: ', format(result.pvalue, '.6f'))
print('p_scientific: ', format(result.pvalue, '.3e'))
print('ci_low: ', round(ci.low, 4))
print('ci_high: ', round(ci.high, 4))

mean_raw:  43.40277777777778
mean_round2:  43.4
mean_format2:  43.40
t_raw:  4.489465664875056
t_round4:  4.4895
p_raw:  7.418465018622554e-05
p_format6:  0.000074
p_scientific:  7.418e-05
ci_low:  41.8641
ci_high:  44.9415


#### 2. 매우 작은 p-value 표기